# Local Learning with Spiking Neural Networks: Estimation

# Outline

## 1) Define populations

* **State layer (X)**: $N_x$ LIF neurons with decoder $D\in R^{K\times N_x}$, recurrency $\Omega_s,\Omega_f$ and yet-to-learn Kalman-gain loop $O_k$.
* **Error layer (E):** $N_e=K$ LIF neurons with decoder $D_e=I_{K\times K}$, leak $\lambda$, and only fast self-reset ($\Omega_f^e=D_e^\top D_e$).

---

## 2) Spiking dynamics each timestep

1. **X-layer step**:

   $$
   \dot v_x = -\lambda v_x + \Omega_s\,r_x + \Omega_f\,s_x  + F_i\,u
               + \underbrace{\Omega_k\,r_x + F_k\,r_e}_{\text{from E}}
               + \eta,
   $$

   threshold $\to$ spike $s_x$, update rate $r_x$.
2. **Decode** $\hat x = D\,r_x$.
3. **Compute prediction error** $\;e = y - C\,\hat x\in R^K$.
4. **E-layer step**:

   $$
   \dot v_e = -\lambda v_e - \Omega_f^e\,s_e + D_e^\top(\,e + \lambda e)\,,
   $$

   threshold $\to$ spike $s_e$, update rate $r_e$.

---

## 3) Local, spike-driven plasticity

After each timestep $t$, use the new filtered rates $r_x(t)$, $r_e(t)$ and the fresh spikes $s_x(t)$, $s_e(t)$ to update:

1. **Observation matrix $C$** (X→E synapses):

   $$
     \Delta C_{i j}
     \;=\;\eta_C\;\bigl[r^{(E)}_i(t)\bigr]\;\bigl[r^{(X)}_j(t)\bigr]
   $$
2. **Kalman gain $K_f$** (E→X synapses):

   $$
     \Delta (K_f)_{p i}
     \;=\;\eta_K\;\bigl[r^{(X)}_p(t)\bigr]\;\bigl[r^{(E)}_i(t)\bigr]
   $$

Immediately **recompute** the dependent loops:

$$
\Omega_k = -\,D^\top K_f\,C\,D,\quad
F_k = D^\top K_f.
$$

All other loops ($\Omega_s,\Omega_f,F_i$) remain fixed.

---

## 4) Simulation in this notebook

* **Initialize** with

  * random small $C,K_f$,
  * random-ish $D$ tiling the state‐space,
  * $\Omega_s=D^\top(A+\lambda I)D$, $\Omega_f=-D^\top D$.
* **Run** for a short horizon (e.g. 1 s at dt=1 ms): at each step we do the two-layer spiking update + plasticity above.
* **Plot**

  1. $\hat x(t)$ (from SCN) vs. true $x(t)$ before/after learning.
  2. Evolution of the learned $C$ or $K_f$ toward their analytic values.
  3. A raster of spikes from X and E layers.

# Installations

In [1]:
import numpy as np

# LIF Layer

This `LIFLayer` class models a small population of leaky integrate-and-fire neurons all at once, keeping track of each neuron’s membrane voltage, its recent firing rate, and whether it just spiked.  Every time you call its `step(...)` method, it first computes how the voltages should drift and interact — letting slow recurrent connections push them up, fast resets knock them down when spikes occur, and any external drive nudge them as well — then it adds a bit of random noise, checks which neuron has crossed its individual threshold, fires exactly one spike if any have, and finally updates each neuron’s smoothed firing rate.  The `decode()` method simply multiplies the current firing rates by a fixed decoder matrix to give you the layer’s "guess" of whatever quantity those neurons are representing.  You can use the same class for both your state-coding layer and your error-coding layer by giving it the appropriate decoder matrix and weight matrices at initialization.

In [4]:
class LIFLayer:
    """
    A generic leaky‐integrate‐and‐fire population.

    Attributes:
        N        : int, number of neurons
        lam      : float, leak rate
        D        : ndarray (K x N), decoder matrix
        Omega_s  : ndarray (N x N), slow recurrent weights
        Omega_f  : ndarray (N x N), fast recurrent weights
        F_in     : ndarray (N x M), feed-forward weights (optional)
        T        : ndarray (N,), spike threshold for each neuron
        v        : ndarray (N,), membrane potentials
        r        : ndarray (N,), filtered spike rates
        s        : ndarray (N,), latest spike indicators (0 or 1/dt)
    """
    def __init__(self, N, lam, D, Omega_s, Omega_f, F_in=None, T=None, dt=1e-3):
        self.N = N
        self.lam = lam
        self.D = D
        self.Omega_s = Omega_s
        self.Omega_f = Omega_f
        self.F_in = F_in
        self.dt = dt

        # Thresholds: default to diag(D.T @ D) / 2 as in the paper
        self.T = (np.diag(D.T @ D) / 2.0) if (T is None) else T

        # State variables
        self.v = np.zeros(N)
        self.r = np.zeros(N)
        self.s = np.zeros(N)

    def step(self, input_drive=None, noise_sigma=0.0):
        """
        Advance the layer by one timestep.

        Args:
            input_drive: ndarray (M,) or None, feed-forward input (u for X, e for E)
            noise_sigma: float, standard deviation of voltage noise
        """
        # Compute drive terms
        drive = -self.lam * self.v + self.Omega_s @ self.r - self.Omega_f @ self.s
        if self.F_in is not None and input_drive is not None:
            drive += self.F_in @ input_drive

        # Euler update with optional noise
        self.v += drive * self.dt + np.sqrt(self.dt) * noise_sigma * np.random.randn(self.N)

        # Thresholding: only the neuron with max v may fire (firing the first one)
        above = np.where(self.v > self.T)[0]
        self.s.fill(0.0)
        if above.size > 0:
            k = above[np.argmax(self.v[above])]
            self.s[k] = 1.0 / self.dt  # Dirac approximation

        # Reset is handled implicitly via the fast term in next drive

        # Update filtered rates
        dr = self.s - self.lam * self.r
        self.r += dr * self.dt

    def decode(self):
        """
        Return the decoded output: x_hat = D @ r.
        """
        x_hat = self.D @ self.r
        return x_hat

# Kalman + SCN

The `KalmanSCN` class implements a fully spiking, recurrent Kalman‐filter estimator using two coupled LIF populations. At each timestep it first propagates a "state" population (X) driven by its internal predictive loop, a fast reset loop, an external control input, and a corrective feedback loop derived from the current estimate of the Kalman gain.  It then decodes those spikes into a state estimate $\hat x$.  Simultaneously, an "error" population (E) receives only feed‐forward spikes encoding the raw measurement $y$ and recurrent inhibition from the X‐layer that implements $-C\hat x$, so that its voltages naturally represent the innovation $y - C\hat x$.  Both populations fire irregular, sparse spikes; their filtered rates $r_x$ and $r_e$ are used in two local Hebbian updates ($\Delta C\propto r_e\otimes r_x$ and $\Delta K_f\propto r_x\otimes r_e$) to adapt the observation matrix and Kalman gain on‐the‐fly.  After each plasticity step the code recomputes the Kalman correction loops in the X‐layer so that all operations remain strictly local, spike‐driven, and biologically plausible.

In [6]:
class KalmanSCN:
    """
    A spiking Kalman filter with two LIF populations (state X and error E) that locally learns
    both the observation matrix C and the Kalman gain Kf via Hebbian spike-driven plasticity.

    Attributes:
        A         : ndarray (K x K), state-transition matrix
        B         : ndarray (K x P), control-input matrix
        C         : ndarray (Q x K), observation matrix (learned)
        Kf        : ndarray (K x Q), Kalman gain matrix (learned)
        lam       : float, leak rate
        dt        : float, timestep size
        eta_C     : float, learning rate for C
        eta_K     : float, learning rate for Kf
        X_layer   : LIFLayer, state-coding LIF population
        E_layer   : LIFLayer, error-coding LIF population
    """
    def __init__(self, A, B, C, Kf, D, D_e, lam, dt, eta_C, eta_K):
        # Store system and learning params
        self.A, self.B = A, B
        self.C = C.copy()
        self.Kf = Kf.copy()
        self.lam = lam
        self.dt = dt
        self.eta_C = eta_C
        self.eta_K = eta_K

        # Dimensions
        K, N_x = D.shape
        N_e = K
        
        # Build SCN loops
        # Predictive & fast loops for X
        Omega_s = D.T @ (A + lam*np.eye(K)) @ D  # by def
        Omega_f = -D.T @ D  # by def
        F_i = D.T @ B  # by def
        Omega_k = -D.T @ Kf @ C @ D  # by def
        F_k = D.T @ Kf  # by def
        T = np.diag(D.T @ D) / 2.0  # by def in the paper
        
        # Identity decoder for E
        Omega_s_e = - D_e.T @ self.C @ D # NOT BY DEF. my attempt at encoding -C*x_hat
        Omega_f_e = -D_e.T @ D_e  # by def
        F_in_e = D_e.T  # almost by def, dropped B. CONSIDER = (1+lam) * D_e.T
        T_e = np.diag(D_e.T @ D_e) / 2.0  # by def in the paper
        
        # Instantiate X and E layers
        self.X_layer = LIFLayer(N_x, lam, D, Omega_s, Omega_f, F_in=np.hstack([F_i, F_k]), T=T, dt=dt)
        self.E_layer = LIFLayer(N_e, lam, D_e, Omega_s_e, Omega_f_e, F_in=F_in_e, T=T_e, dt=dt)

    def step(self, u, y):
           """
        Advance the network by one timestep: update state and error layers, apply plasticity.

        Args:
            u : ndarray (P,), control input at current timestep
            y : ndarray (Q,), measurement input at current timestep

        Returns:
            x_hat : ndarray (K,), current state estimate after update
        """
        # State layer update with combined input [u; r_e]
        input_x = np.concatenate([u, self.E_layer.r])
        self.X_layer.step(input_x)
        
        # Decode state estimate
        x_hat = self.X_layer.decode()
        
        # Compute innovation
        # e = y - self.C @ x_hat -> using this to update the Error layer would be cheating, right?
        
        # Error layer update
        self.E_layer.step(y)
        
        # Apply plasticity and rebuild loops
        self.update_weights()
        self.rebuild_kalman()
        
        return x_hat, e

    def update_weights(self):
        """
        Apply local Hebbian updates to the observation matrix C and Kalman gain Kf.
        """
        # Local Hebbian updates
        r_x = self.X_layer.r
        r_e = self.E_layer.r
        
        self.C += self.eta_C * np.outer(r_e, r_x)  # ΔC_ij ∝ e_i xhat_j
        self.Kf += self.eta_K * np.outer(r_x, r_e)  # reasoning: Le = x_hat ∝ r_x, and e ∝ r_e

    def rebuild_kalman(self):
        """
        Recompute the Kalman correction loops based on updated C and Kf.
        """
        # Recompute Kalman loops in X_layer
        D = self.X_layer.D
        C, Kf = self.C, self.Kf
        
        Omega_k = -D.T @ Kf @ C @ D
        F_k = D.T @ Kf
        
        # Update X_layer's feedforward matrix [F_i, *F_k*]
        self.X_layer.F_in = np.hstack([self.F_i, F_k])
        # Update X_layer's recurrent slow loop (Omega_s + *Omega_k*)
        self.X_layer.Omega_s = self.Omega_s + Omega_k

IndentationError: unindent does not match any outer indentation level (<string>, line 63)